## Dataset Process

In [ ]:
from datasets import load_dataset

dataset = load_dataset('parquet', data_files='../../raw_data/NMRExp/NMRexp_10to24_1_1004.parquet')

In [ ]:
from tqdm import tqdm

data_list = []
smiles_to_index = {}  # O(1) lookup instead of O(n)
nmr_fields = ['NMR_frequency', 'NMR_solvent', 'NMR_shift_text', 
              'NMR_note', 'NMR_processed', 'Atom_number', 
              'Atom_number_diff_env', 'Atom_number_abstract']
metadata_fields = ['Filename', 'SMILES', 'Page_in_file_mol', 
                   'Page_in_file_para', 'Location_in_page_mol', 
                   'Location_in_page_para']

for i in tqdm(range(len(dataset['train']))):
    data = dataset['train'][i]
    smiles = data['SMILES']
    
    if smiles not in smiles_to_index:
        # First occurrence: create new entry
        smiles_to_index[smiles] = len(data_list)
        data_to_merge = {key: data[key] for key in metadata_fields}
        data_to_merge[data['NMR_type']] = {key: data[key] for key in nmr_fields}
        data_list.append(data_to_merge)
    else:
        # Duplicate: merge NMR_type if not present
        index = smiles_to_index[smiles]
        if data['NMR_type'] not in data_list[index]:
            data_list[index][data['NMR_type']] = {key: data[key] 
                                                   for key in nmr_fields}


In [ ]:
from datasets import Dataset
dataset = Dataset.from_list(data_list)
dataset.to_parquet('../../raw_data/NMRExp/NMRExp_10to24_1_1004_merged.parquet')

In [ ]:
from datasets import Dataset
data_CH_list = []
for data in data_list:
    if '13C NMR' in data and '1H NMR' in data:
        data_CH_list.append({key: data[key] for key in metadata_fields + ['13C NMR', '1H NMR']})
dataset_CH = Dataset.from_list(data_CH_list)
dataset_CH.to_parquet('../../raw_data/NMRExp/NMRExp_10to24_1_1004_merged_C&H.parquet')

In [ ]:
import os
def filter_CH(example):
    if example['13C NMR'] is None or example['1H NMR'] is None:
        return False
    return True
dataset_CH = dataset.filter(filter_CH)

# dataset_CH.to_parquet('../../raw_data/NMRExp/NMRExp_10to24_1_1004_merged_C&H.parquet')

In [ ]:
from datasets import load_dataset

dataset_CH = load_dataset('parquet', data_files='../../raw_data/NMRExp/NMRExp_10to24_1_1004_merged_C&H.parquet')['train']

In [ ]:
dataset = load_dataset('parquet', data_files='../../raw_data/NMRExp/NMRexp_10to24_1_1004_merged_C&H_converted.parquet')['train']

In [ ]:
from rdkit import Chem, rdBase
import logging
from io import StringIO
from rdkit.Chem import rdMolDescriptors
from ast import literal_eval
from tqdm import tqdm

# route RDKit C++ logs to Python logger
rdBase.LogToPythonLogger()
logger = logging.getLogger("rdkit")

stream = StringIO()
handler = logging.StreamHandler(stream)
formatter = logging.Formatter("%(levelname)s: %(message)s")
handler.setFormatter(formatter)
handler.setLevel(logging.INFO)
logger.addHandler(handler)

data_list = []
error_list = []
error_idx_list = []

for i in tqdm(range(len(dataset_CH))):
    try:
        entry = dataset_CH[i]

        # reset log capture
        stream.seek(0)
        stream.truncate(0)

        smiles = entry['SMILES']
        mol = Chem.MolFromSmiles(smiles)

        # read RDKit messages for this molecule
        log_text = stream.getvalue()

        # check for the specific conflict message
        if "Conflicting single bond directions around double bond" in log_text:
            error_list.append(log_text.strip())
            error_idx_list.append(i)
            raise ValueError(f"RDKit Warning for SMILES {smiles} at index {i}")

        # your existing processing
        molecular_formula = rdMolDescriptors.CalcMolFormula(mol)
        num_C = entry['13C NMR']['Atom_number']
        num_H = entry['1H NMR']['Atom_number']
        nmr_13C = literal_eval(entry['13C NMR']['NMR_processed'])
        nmr_1H = literal_eval(entry['1H NMR']['NMR_processed'])

        c_nmr_peaks = []
        for peak in nmr_13C:
            if isinstance(peak[0], list):
                peak_pos = round(sum(peak[0]) / len(peak[0]), 2)
            else:
                peak_pos = peak[0]
            c_nmr_peaks.append({
                'delta (ppm)': peak_pos,
                'integral': None,
                'intensity': None,
                'width (ppm)': None
            })

        h_nmr_peaks = []
        for peak in nmr_1H:
            peak_type = peak[0]
            if peak[1]:
                try:
                    j_values = '_'.join([j.replace('Hz', '') for j in literal_eval(peak[1])]) + '_'
                except:
                    j_values = '_'.join([j.replace('Hz', '') for j in peak[1]]) + '_'
            else:
                j_values = None
            h_counts = int(peak[2].replace('H', '')) if peak[2] else 1
            max_shift = peak[3]
            min_shift = peak[4]
            h_nmr_peaks.append({
                'category': peak_type,
                'centroid': round((max_shift + min_shift) / 2, 2),
                'delta': round((max_shift + min_shift) / 2, 2),
                'j_values': j_values,
                'nH': h_counts,
                'rangeMax': max_shift,
                'rangeMin': min_shift
            })

        data_list.append({
            'idx': i,
            'smiles': smiles,
            'molecular_formula': molecular_formula,
            'num_C': num_C,
            'num_H': num_H,
            'c_nmr_peaks': c_nmr_peaks,
            'h_nmr_peaks': h_nmr_peaks,
            'num_c_peaks': len(c_nmr_peaks),
            'num_h_peaks': sum(peak['nH'] for peak in h_nmr_peaks)
        })
    except Exception as e:
        error_list.append(str(e))
        error_idx_list.append(i)

In [ ]:
from datasets import Dataset
dataset_CH = Dataset.from_list(data_list)
dataset_CH.to_parquet('../../raw_data/NMRExp/NMRexp_10to24_1_1004_merged_C&H_converted.parquet')

## NMR2Mol Dataset Generation

In [ ]:
from datasets import load_dataset
from nmr_rise.utils.data import filter_invalid_smiles
from datasets import DatasetDict
dataset = load_dataset('parquet', data_files='../../raw_data/NMRExp/NMRexp_10to24_1_1004_merged_C&H_converted.parquet')['train']
print("Dataset size before filtering invalid SMILES:", len(dataset))
dataset = dataset.filter(filter_invalid_smiles, num_proc=16)
print("Dataset size after filtering invalid SMILES:", len(dataset))
def canonicalize_smiles(entry):
    """Convert a SMILES string to its canonical form."""
    mol = Chem.MolFromSmiles(entry['smiles'])
    if mol is None:
        entry['smiles'] = None
    else:
        entry['smiles'] = Chem.MolToSmiles(mol)
    return entry

def filter_metal_atoms(entry):
    mol = Chem.MolFromSmiles(entry['smiles'])
    metal_atomic_numbers = set(range(3, 104)) - {1, 2, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18,
                                                 19, 20, 31, 32, 33, 34, 35, 36, 37, 38, 49, 50,
                                                 51, 52, 53, 54, 55, 56, 81, 82, 83, 84, 85, 86,
                                                 87, 88}
    for atom in mol.GetAtoms():
        if atom.GetAtomicNum() in metal_atomic_numbers:
            return False
    return True

def filter_invalid_hnmr(entry):
    hnmr_peaks = entry['h_nmr_peaks']
    for peak in hnmr_peaks:
        try:
            range_max = float(peak["rangeMax"]) 
            range_min = float(peak["rangeMin"]) 

            formatted_peak = ""
            formatted_peak = formatted_peak + "{:.2f} {:.2f} ".format(range_max, range_min)        
            formatted_peak = formatted_peak +  "{} {}H ".format(
                                                                peak["category"],
                                                                peak["nH"],
                                                            )
            js = str(peak["j_values"])
            if js != "None":
                split_js = js.split("_")
                split_js = list(filter(None, split_js))
                processed_js = ["{:.2f}".format(float(j)) for j in split_js]
                formatted_js = "J " + " ".join(processed_js)
                formatted_peak += formatted_js
        except:
            return False
    return True


# split dataset into train, val, test with 80% train, 10% val, 10% test
def cal_max_shift(entry):
    cnmr_values = [peak['delta (ppm)'] for peak in entry['c_nmr_peaks']]
    hnmr_values = [peak['delta'] for peak in entry['h_nmr_peaks']]
    max_c_shift = max(cnmr_values) if cnmr_values else 0
    max_h_shift = max(hnmr_values) if hnmr_values else 0
    min_c_shift = min(cnmr_values) if cnmr_values else 0
    min_h_shift = min(hnmr_values) if hnmr_values else 0
    entry['max_c_shift'] = max_c_shift
    entry['max_h_shift'] = max_h_shift
    entry['min_c_shift'] = min_c_shift
    entry['min_h_shift'] = min_h_shift
    return entry
dataset = dataset.map(cal_max_shift, num_proc=16)
print("Dataset size before filtering invalid shifts and canonicalization:", len(dataset))
dataset = dataset.filter(lambda x: x['max_c_shift'] <= 250 and x['max_h_shift'] <= 15, num_proc=16)
dataset = dataset.filter(lambda x: x['min_c_shift'] >= -20 and x['min_h_shift'] >= -2, num_proc=16)
dataset = dataset.map(canonicalize_smiles, num_proc=16)
dataset = dataset.filter(lambda x: x['smiles'] is not None, num_proc=16)
print("Dataset size after filtering invalid shifts and canonicalization:", len(dataset))

print("Dataset size after filtering invalid hnmr:", len(dataset))
# dataset = dataset.filter(filter_metal_atoms, num_proc=16)
dataset = dataset.filter(filter_invalid_hnmr, num_proc=16)
print("Dataset size after filtering invalid hnmr:", len(dataset))

train_test_split = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = train_test_split['train']
test_valid_dataset = train_test_split['test']
val_test_split = test_valid_dataset.train_test_split(test_size=0.5, seed=42)
val_dataset = val_test_split['train']
test_dataset = val_test_split['test']
dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})
dataset.save_to_disk('../../data/NMRExp_1/full')

## Mol2NMR Dataset Generation

In [ ]:
from datasets import load_from_disk
from nmr_rise.utils.data import conformation_construction, filter_invalid_entry
dataset = load_from_disk('../../data/NMRExp_1/full')
for split in ['train', 'validation', 'test']:
    dataset[split] = dataset[split].map(conformation_construction, num_proc=16)
    dataset[split] = dataset[split].filter(filter_invalid_entry, num_proc=16)
dataset.save_to_disk('../../data/NMRExp_1/mol2nmr/full_cc')

## NMR2Mol Dataset Generation

In [ ]:
import os
from datasets import load_from_disk
import json
import ast
import copy
def load_nmr2mol_pred_results(dir_path, filename):
    with open(os.path.join(dir_path, filename), 'r') as f:
        return {item['true']: str(item['pred']) for item in json.load(f)}

def add_candidates(entry, lookup, validity_check=True):
    candidates = ast.literal_eval(lookup.get(entry['smiles'], ""))

    if validity_check:
        valid_candidates = []
        for cand in candidates:
            confs = conformation_construction({'smiles': copy.deepcopy(cand)})
            if confs['is_converted']:
                valid_candidates.append(cand)
        entry['candidates'] = valid_candidates
    else:
        entry['candidates'] = candidates
    return entry

def molref_data_gen(data_path, dataset_name, nmr2mol_pred_dir):
    dataset = load_from_disk(os.path.join(data_path, dataset_name))
    nmr2mol_pred_path = os.path.join(data_path, nmr2mol_pred_dir)
    lookups = {
        'train': load_nmr2mol_pred_results(nmr2mol_pred_path, 'train_split_result.json'),
        'validation': load_nmr2mol_pred_results(nmr2mol_pred_path, 'valid_split_result.json'),
        'test': load_nmr2mol_pred_results(nmr2mol_pred_path, 'test_split_result.json')
    }
    for split in ['train', 'validation', 'test']:
        print(len(lookups[split]))
        dataset[split] = dataset[split].map(lambda ex: add_candidates(ex, lookups[split], validity_check=False))
    return dataset

In [ ]:
dataset = molref_data_gen('../../data/NMRExp', 'full', 'nmr2mol_pred')

In [ ]:
import random
from ast import literal_eval
def candidate_aug(batch, aug_num=5):
    """
    Batch version of candidate augmentation function.
    For each example in the batch, samples aug_num candidates from the 'candidates' list,
    then expands each example into multiple rows - one per sampled candidate.
    
    Args:
        batch (dict): A batch of examples as lists for each column.
        aug_num (int): Number of candidates to sample and expand per example.
    
    Returns:
        dict: Augmented batch with each example expanded into multiple rows.
    """
    aug_rows = {
        k: [] for k in batch if k != 'candidates'
    }
    aug_rows['candidate'] = []
    aug_rows['aug_id'] = []
    
    for i, candidates in enumerate(batch['candidates']):
        # Sample candidates with or without replacement
        sampled_candidates = random.sample(candidates, aug_num)
        
        for j, candidate in enumerate(sampled_candidates):
            # Copy all columns except 'candidates'
            for key in batch:
                if key != 'candidates':
                    aug_rows[key].append(batch[key][i])
            aug_rows['candidate'].append(candidate)
            aug_rows['aug_id'].append(f"aug_{batch.get('idx', [''])[i]}_{j}")

    return aug_rows

def dataset_aug_molref(dataset, aug_num=5, aug_max=False):
    """
    Augments the dataset for training the MolRef model by expanding each example into multiple rows based on candidates.
    
    Args:
        dataset (datasets.Dataset): HuggingFace dataset to augment.
        aug_num (int): Number of augmented samples per example.
    
    Returns:
        datasets.Dataset: Augmented dataset with expanded rows and extra columns.
    """
    return dataset.map(
        lambda batch: candidate_aug(batch, aug_num=len(batch['candidates'][0]) if aug_max else min(aug_num, len(batch['candidates'][0]))),
        batched=True,
        batch_size=1,  # Process one example at a time for expansion
        remove_columns=['candidates'],
        num_proc=os.cpu_count()
    )


In [ ]:
from datasets import DatasetDict
for aug_num in [1, 3, 5, 10]:
    shuffle_seed = 42
    train_aug = dataset_aug_molref(dataset['train'], aug_num=aug_num, aug_max=False)
    val_aug = dataset_aug_molref(dataset['validation'], aug_num=aug_num, aug_max=False)
    test_aug = dataset_aug_molref(dataset['test'], aug_num=aug_num, aug_max=False)
    train_aug = train_aug.shuffle(seed=shuffle_seed)
    val_aug = val_aug.shuffle(seed=shuffle_seed)
    test_aug = test_aug.shuffle(seed=shuffle_seed)
    aug_dataset = DatasetDict({
        'train': train_aug,
        'validation': val_aug,
        'test': test_aug
    })
    aug_dataset.save_to_disk(os.path.join('../../data/NMRExp', f'revision_{aug_num}'))

# Evaluation Dataset Generation

In [ ]:
from datasets import load_from_disk
import random

test_dataset = load_from_disk('../../data/NMRExp/full')['test']

# randomly select 1000 samples from test_dataset
random.seed(42)
selected_indices = random.sample(range(len(test_dataset)), 10000)
selected_samples = test_dataset.select(selected_indices)
selected_samples.save_to_disk('../../data/NMRExp/NMRExp-10000')

selected_indices = random.sample(range(len(selected_samples)), 1000)
selected_samples = selected_samples.select(selected_indices)
selected_samples.save_to_disk('../../data/NMRExp/NMRExp-1000')